# Mini Assessment – Feature Engineering & Feature Selection

## Objective

The objective of this mini assessment is to apply the complete feature engineering and feature selection workflow to a telecom customer churn prediction problem.

Customer churn occurs when a customer stops using a company's services. Predicting churn helps businesses identify customers who may leave and take preventive actions.

## Concepts Covered

1. Data inspection and understanding
2. Numerical feature engineering
3. Categorical feature engineering
4. Domain-based feature creation
5. Interaction features
6. Feature selection
7. Feature importance
8. Data leakage prevention
9. Feature engineering pipelines
10. Model training and evaluation

## Expected Learning Outcome

After completing this notebook, I should be able to:

- Identify useful information from raw data.
- Create meaningful features using existing columns.
- Explain why each feature is created.
- Select relevant features for machine learning.
- Prevent target leakage.
- Build a preprocessing and machine learning pipeline.
- Evaluate and compare model performance.

## Business Problem

A telecom company wants to predict whether a customer will leave its service.

The target variable is `Churn`:

- 0 – Customer stays.
- 1 – Customer leaves.

The goal is to prepare useful machine learning features and build a churn prediction model.

# Mini Assessment Tasks

We will use a telecom customer dataset containing customer information, service details, and billing information.

## Assessment Questions

**Question 1: Data Understanding**

- What is the shape of the dataset?
- Are there missing values?
- What is the target variable?

**Question 2: Feature Engineering**

Create meaningful features from the existing columns:

- `Average_Monthly_Spending`
- `Service_Count`
- `Customer_Loyalty_Group`
- `Support_Risk_Level`
- `Long_Term_Customer`
- `Tenure_Charge_Interaction`

**Question 3: Feature Selection**

Identify useful features for predicting customer churn using statistical feature selection.

**Question 4: Leakage Prevention**

Check whether any feature directly reveals the target variable. Ensure that the target variable is not included in the input features.

**Question 5: Machine Learning Pipeline**

Build a pipeline containing:

- Missing-value handling
- Numerical scaling
- Categorical encoding
- Feature selection
- Classification model

**Question 6: Model Evaluation**

Evaluate the model using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

**Question 7: Final Analysis**

Compare the model using original features with the model using engineered features.

Explain whether feature engineering improved the model and identify the features selected by the feature selection process.

In [1]:
# ==========================================================
# NOTEBOOK 17: MINI ASSESSMENT
# Feature Engineering & Feature Selection
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ----------------------------------------------------------
# 1. CREATE A SAMPLE TELECOM CUSTOMER DATASET
# ----------------------------------------------------------

np.random.seed(42)

n = 600

data = pd.DataFrame({
    "Age": np.random.randint(18, 70, n),
    "Tenure": np.random.randint(1, 73, n),
    "MonthlyCharges": np.random.uniform(20, 120, n),
    "SupportCalls": np.random.poisson(2, n),
    "ContractType": np.random.choice(
        ["Month-to-month", "One year", "Two year"], n
    ),
    "InternetService": np.random.choice(
        ["DSL", "Fiber optic", "No"], n
    ),
    "PhoneService": np.random.choice(["Yes", "No"], n),
    "OnlineSecurity": np.random.choice(["Yes", "No"], n)
})

# Generate TotalCharges from tenure and monthly charges
data["TotalCharges"] = (
    data["Tenure"] * data["MonthlyCharges"]
)

# Generate churn using realistic business signals
churn_score = (
    0.8
    - 0.025 * data["Tenure"]
    + 0.20 * data["SupportCalls"]
    + 0.015 * data["MonthlyCharges"]
    + (data["ContractType"] == "Month-to-month") * 1.0
    - (data["ContractType"] == "Two year") * 0.8
)

churn_probability = 1 / (1 + np.exp(-churn_score))

data["Churn"] = (
    np.random.random(n) < churn_probability
).astype(int)

# Introduce missing values
for column in ["Age", "MonthlyCharges", "SupportCalls"]:
    missing_indices = data.sample(frac=0.03, random_state=42).index
    data.loc[missing_indices, column] = np.nan

print("DATASET CREATED SUCCESSFULLY")
print("-" * 50)
print("Dataset Shape:", data.shape)
print("\nFirst Five Records:")
print(data.head())

print("\nMissing Values:")
print(data.isnull().sum())

print("\nTarget Distribution:")
print(data["Churn"].value_counts())


# ----------------------------------------------------------
# 2. SPLIT DATA INTO TRAINING AND TESTING SETS
# ----------------------------------------------------------

X = data.drop(columns=["Churn"])
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTRAIN-TEST SPLIT")
print("-" * 50)
print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)


# ----------------------------------------------------------
# 3. CREATE ENGINEERED FEATURES
# ----------------------------------------------------------

def engineer_features(df):

    df = df.copy()

    # Numerical feature: average monthly spending
    df["Average_Monthly_Spending"] = (
        df["TotalCharges"] / df["Tenure"].replace(0, np.nan)
    )

    # Numerical feature: number of subscribed services
    df["Service_Count"] = (
        (df["InternetService"] != "No").astype(int)
        + (df["PhoneService"] == "Yes").astype(int)
        + (df["OnlineSecurity"] == "Yes").astype(int)
    )

    # Categorical feature: customer loyalty group
    df["Customer_Loyalty_Group"] = pd.cut(
        df["Tenure"],
        bins=[0, 12, 36, 72],
        labels=["New", "Regular", "Loyal"],
        include_lowest=True
    ).astype("object")

    # Categorical feature: support risk level
    df["Support_Risk_Level"] = pd.cut(
        df["SupportCalls"],
        bins=[-1, 1, 3, np.inf],
        labels=["Low", "Medium", "High"]
    ).astype("object")

    # Numerical feature: long-term customer indicator
    df["Long_Term_Customer"] = (
        df["Tenure"] >= 36
    ).astype(int)

    # Numerical interaction feature
    df["Tenure_Charge_Interaction"] = (
        df["Tenure"] * df["MonthlyCharges"]
    )

    return df


X_train_engineered = engineer_features(X_train)
X_test_engineered = engineer_features(X_test)

print("\nFEATURE ENGINEERING COMPLETED")
print("-" * 50)
print("Original Training Shape:", X_train.shape)
print("Engineered Training Shape:", X_train_engineered.shape)

print("\nEngineered Features:")
print([
    "Average_Monthly_Spending",
    "Service_Count",
    "Customer_Loyalty_Group",
    "Support_Risk_Level",
    "Long_Term_Customer",
    "Tenure_Charge_Interaction"
])


# ----------------------------------------------------------
# 4. DEFINE A REUSABLE MACHINE LEARNING PIPELINE
# ----------------------------------------------------------

def create_model_pipeline(X_data):

    numerical_columns = X_data.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_columns = X_data.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    numerical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(
            score_func=f_classif,
            k=12
        )),
        ("model", LogisticRegression(max_iter=1000))
    ])

    return model_pipeline


# ----------------------------------------------------------
# 5. TRAIN AND EVALUATE ORIGINAL-FEATURE MODEL
# ----------------------------------------------------------

original_model = create_model_pipeline(X_train)

original_model.fit(X_train, y_train)

original_predictions = original_model.predict(X_test)

original_accuracy = accuracy_score(
    y_test, original_predictions
)

original_f1 = f1_score(
    y_test, original_predictions, zero_division=0
)

print("\nORIGINAL FEATURE MODEL RESULTS")
print("-" * 50)
print("Accuracy:", round(original_accuracy, 4))
print("F1-score:", round(original_f1, 4))


# ----------------------------------------------------------
# 6. TRAIN AND EVALUATE ENGINEERED-FEATURE MODEL
# ----------------------------------------------------------

engineered_model = create_model_pipeline(
    X_train_engineered
)

engineered_model.fit(
    X_train_engineered, y_train
)

engineered_predictions = engineered_model.predict(
    X_test_engineered
)

accuracy = accuracy_score(
    y_test, engineered_predictions
)

precision = precision_score(
    y_test, engineered_predictions, zero_division=0
)

recall = recall_score(
    y_test, engineered_predictions, zero_division=0
)

f1 = f1_score(
    y_test, engineered_predictions, zero_division=0
)

cm = confusion_matrix(
    y_test, engineered_predictions
)

print("\nENGINEERED FEATURE MODEL RESULTS")
print("-" * 50)
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    engineered_predictions,
    zero_division=0
))


# ----------------------------------------------------------
# 7. IDENTIFY SELECTED FEATURES
# ----------------------------------------------------------

preprocessor = engineered_model.named_steps["preprocessor"]

feature_selector = engineered_model.named_steps[
    "feature_selection"
]

all_feature_names = preprocessor.get_feature_names_out()

selected_features = all_feature_names[
    feature_selector.get_support()
]

print("\nSELECTED FEATURES")
print("-" * 50)

for feature in selected_features:
    print(feature)

print("\nNumber of Selected Features:", len(selected_features))


# ----------------------------------------------------------
# 8. COMPARE ORIGINAL AND ENGINEERED MODELS
# ----------------------------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "Original Features",
        "Engineered Features"
    ],
    "Accuracy": [
        original_accuracy,
        accuracy
    ],
    "F1_Score": [
        original_f1,
        f1
    ]
})

print("\nMODEL COMPARISON")
print("-" * 50)
print(comparison)

if f1 > original_f1:
    print("\nEngineered features achieved a higher F1-score.")
elif f1 < original_f1:
    print("\nOriginal features achieved a higher F1-score.")
else:
    print("\nBoth models achieved the same F1-score.")


# ----------------------------------------------------------
# 9. FEATURE LEAKAGE CHECK
# ----------------------------------------------------------

print("\nFEATURE LEAKAGE CHECK")
print("-" * 50)

if "Churn" not in X_train_engineered.columns:
    print("PASS: Target variable is not included in input features.")
else:
    print("WARNING: Target leakage detected!")

print(
    "Target column used separately:",
    "Churn" not in X_train_engineered.columns
)


# ----------------------------------------------------------
# 10. FINAL ASSESSMENT SUMMARY
# ----------------------------------------------------------

print("\nFINAL ASSESSMENT SUMMARY")
print("=" * 50)

print("1. Dataset inspection completed.")
print("2. Missing values handled using pipeline imputation.")
print("3. Numerical and categorical features engineered.")
print("4. Feature selection performed using SelectKBest.")
print("5. Target leakage checked.")
print("6. Machine learning pipeline trained.")
print("7. Model evaluation completed.")
print("8. Original and engineered models compared.")

print("\nSPRINT 6 MINI ASSESSMENT COMPLETED!")

DATASET CREATED SUCCESSFULLY
--------------------------------------------------
Dataset Shape: (600, 10)

First Five Records:
    Age  Tenure  MonthlyCharges  SupportCalls    ContractType InternetService  \
0  56.0      21       62.142455           2.0        One year              No   
1  69.0      36      109.267171           6.0  Month-to-month     Fiber optic   
2  46.0      10      101.744356           1.0        Two year     Fiber optic   
3  32.0      24       54.181735           3.0  Month-to-month     Fiber optic   
4  60.0      64       45.942343           0.0  Month-to-month     Fiber optic   

  PhoneService OnlineSecurity  TotalCharges  Churn  
0          Yes             No   1304.991556      1  
1           No             No   3933.618160      1  
2           No             No   1017.443562      1  
3          Yes            Yes   1300.361644      0  
4           No             No   2940.309974      1  

Missing Values:
Age                18
Tenure              0
MonthlyC

# Feature Justification and Final Conclusion

## Feature Justification

| Feature Name | Source Columns | Reason for Creation | ML Value |
|---|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | Understand average customer spending | Helps identify spending patterns |
| Service_Count | InternetService, PhoneService, OnlineSecurity | Measure the number of subscribed services | Represents service engagement |
| Customer_Loyalty_Group | Tenure | Group customers based on relationship duration | Identifies customer loyalty |
| Support_Risk_Level | SupportCalls | Categorize customer support requirements | Helps identify potential dissatisfaction |
| Long_Term_Customer | Tenure | Identify customers with longer relationships | Represents customer retention |
| Tenure_Charge_Interaction | Tenure, MonthlyCharges | Understand the combined effect of tenure and charges | Captures feature interaction |

## Feature Selection

SelectKBest with the ANOVA F-test is used to identify statistically useful features.

Only the selected features are passed to the classification model.

Feature selection is performed inside the machine learning pipeline, so the selector learns from training data only.

## Leakage Prevention

- The target column `Churn` is separated from input features.
- No feature is directly calculated using the target variable.
- The train-test split is performed before model training.
- Imputation, scaling, encoding, and feature selection are fitted within the training pipeline.

## Final Conclusion

In this mini assessment, I applied the complete feature engineering and feature selection workflow to a telecom customer churn prediction problem.

I created numerical, categorical, domain-based, and interaction features from raw customer information.

I used a machine learning pipeline to handle missing values, scale numerical data, encode categorical data, and select useful features.

Finally, I trained a Logistic Regression model and evaluated its performance using accuracy, precision, recall, F1-score, and a confusion matrix.

I also compared the model using original features with the model using engineered features.

**Important Learning:** Feature engineering is not simply about creating more columns. It is about creating meaningful information that helps a machine learning model understand the problem better. A new feature should be retained only when it provides useful information without introducing data leakage.